[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/35.%20The%20Terminal%20Layout%20Design%20Problem/P35-Tier-7_executed.ipynb)

---

# 35. The Terminal Layout Design Problem

## Tier 7: Human–AI Collaboration

### Goal

This notebook demonstrates a human–AI decision-support workflow for an illustrative terminal layout. A program generates candidate layouts, calculates simplified performance indicators, ranks candidates using an expert's stated preferences, explains a recommendation, and applies explicit human feedback to produce a refined layout.

### Inputs and outputs

The inputs are a fixed set of facility coordinates, simplified geometric assumptions, expert preference weights, and a feedback record. The outputs are ranked recommendations, metric summaries, a feedback-informed layout, and visual comparisons. The code does not solve a continuous optimization problem or establish that a layout is safe, compliant, or operationally feasible.

### Scope and modeling assumptions

- Coordinates are two-dimensional points measured in meters inside a rectangular terminal area. Distances are Euclidean distances between facility reference points; they are not route lengths or aisle widths.
- The candidate generator uses a small, deterministic family of layouts based on a common facility template. The expert-weighted score ranks these candidates; it does not prove global optimality.
- Throughput, transport efficiency, estimated cost, safety, flexibility, environmental impact, and regulatory compliance are illustrative indicators defined by the code. They are not calibrated against operational data or regulations.
- Human feedback is supplied as an example input. The refinement rules are transparent coordinate adjustments, not a substitute for engineering review.
- The Hamburg setting mentioned in the example is contextual only. No site-specific measurements, verified case-study outcomes, or stakeholder survey results are provided here.

### Method and learning sequence

The workflow follows the same sequence throughout: represent solutions and experts, generate candidate layouts, calculate metrics and explanations, rank candidates using expert preferences, record feedback, apply the specified refinements, and compare the resulting indicators. The final demonstration records outcomes across simulated sessions; it does not train or update a predictive model.

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

# Fixed seeds preserve reproducibility if randomized elements are added to an exercise.
np.random.seed(42)
random.seed(42)
plt.style.use("default")

## 1. Representing solutions, experts, and sessions

A layout solution stores both its facility coordinates and the indicators calculated from them. Separating these pieces makes it possible to compare candidates consistently and to recalculate their indicators after a coordinate change. An expert stores preference weights, while a collaboration session keeps recommendations and feedback together.

In this notebook, a preference weight expresses the relative importance an expert assigns to a criterion. The weights are used in a weighted-sum ranking score; they are not probabilities and do not make subjective preferences objectively correct.

In [ ]:
@dataclass
class LayoutSolution:
    """A candidate layout and the indicators calculated for it."""

    facility_assignments: Dict[str, Tuple[float, float]]
    quantitative_metrics: Dict[str, float]
    qualitative_factors: Dict[str, float]
    ai_confidence: float
    human_acceptance: Optional[float] = None
    explanation: Optional[Dict[str, Any]] = None


@dataclass
class HumanExpert:
    """An expert profile with areas of expertise and criterion weights."""

    name: str
    expertise_areas: List[str]
    experience_years: int
    decision_weights: Dict[str, float]


@dataclass
class AIRecommendation:
    """A ranked candidate together with its explanation and alternatives."""

    solution: LayoutSolution
    primary_objective: str
    feature_importance: Dict[str, float]
    reasoning: List[str]
    alternatives: List[LayoutSolution]
    confidence_score: float


@dataclass
class CollaborationSession:
    """The recommendations and feedback associated with one review session."""

    session_id: str
    human_expert: HumanExpert
    ai_recommendations: List[AIRecommendation]
    human_feedback: List[Dict[str, Any]]
    final_solution: Optional[LayoutSolution] = None
    collaboration_metrics: Dict[str, float] = field(default_factory=dict)


print("Collaboration data structures are ready.")

## 2. Candidate generation, metrics, and explanations

The candidate generator creates a small, deterministic set of layouts by varying the yard-block row around a common template. This supplies alternatives for the ranking step without implying that every feasible layout has been explored.

For a candidate, let $B$, $Y$, and $G$ denote the sets of berth, yard, and gate reference points. The notebook uses mean pairwise Euclidean distances:

$$
\bar d_{BY} = \frac{1}{|B||Y|} \sum_{b \in B} \sum_{y \in Y} \lVert b-y \rVert_2,
\qquad
\bar d_{YG} = \frac{1}{|Y||G|} \sum_{y \in Y} \sum_{g \in G} \lVert y-g \rVert_2.
$$

The code defines the throughput indicator as $1000/(\bar d_{BY}+\bar d_{YG})$ and the transport-efficiency indicator as $100/(\bar d_{BY}+\bar d_{YG}+1)$. These are dimension-dependent proxies, not measured throughput or percentages of real-world efficiency. The estimated cost is a simplified distance-weighted quantity. Qualitative indicators are also explicit proxies: for example, the safety score depends on the minimum distance between facility reference points, not on aisle geometry or a safety standard.

The confidence score is a fixed weighted combination of normalized throughput, safety, and transport-efficiency indicators. It summarizes those selected indicators only; it is not a statistically calibrated probability.

In [ ]:
class ExplainableAIOptimizer:
    """Generate, score, and explain illustrative terminal layout candidates."""

    def __init__(self):
        self.terminal_width = 1000.0
        self.terminal_length = 800.0
        self.edge_buffer = 50.0

    def generate_ai_solutions(self, num_solutions: int = 10) -> List[LayoutSolution]:
        """Generate a deterministic family of candidates and calculate their indicators."""
        if num_solutions <= 0:
            raise ValueError("num_solutions must be positive")

        solutions = []
        for variant in range(num_solutions):
            assignments = self._generate_facility_layout(variant)
            quantitative = self._calculate_quantitative_metrics(assignments)
            qualitative = self._calculate_qualitative_factors(assignments)
            confidence = self._calculate_ai_confidence(quantitative, qualitative)
            solutions.append(
                LayoutSolution(
                    facility_assignments=assignments,
                    quantitative_metrics=quantitative,
                    qualitative_factors=qualitative,
                    ai_confidence=confidence,
                )
            )
        return solutions

    def _generate_facility_layout(
        self, variant: int = 0
    ) -> Dict[str, Tuple[float, float]]:
        """Create one template layout with a deterministic yard-row variation."""
        yard_y = 280.0 + 20.0 * (variant % 5)
        maintenance_x = 490.0 + 10.0 * (variant % 3)

        return {
            "berth_1": (50.0, 0.0),
            "berth_2": (450.0, 0.0),
            "berth_3": (850.0, 0.0),
            "yard_block_1": (200.0, yard_y),
            "yard_block_2": (500.0, yard_y),
            "yard_block_3": (800.0, yard_y),
            "gate_1": (100.0, 700.0),
            "gate_2": (500.0, 700.0),
            "gate_3": (900.0, 700.0),
            "maintenance": (maintenance_x, 150.0),
            "rail_terminal": (50.0, 500.0),
            "admin_building": (500.0, 500.0),
        }

    @staticmethod
    def _euclidean_distance(
        point_a: Tuple[float, float], point_b: Tuple[float, float]
    ) -> float:
        """Return straight-line distance between two coordinate points."""
        return float(np.hypot(point_a[0] - point_b[0], point_a[1] - point_b[1]))

    def _calculate_quantitative_metrics(
        self, assignments: Dict[str, Tuple[float, float]]
    ) -> Dict[str, float]:
        """Calculate the notebook's geometric proxy indicators."""
        berth_coords = [value for name, value in assignments.items() if "berth" in name]
        yard_coords = [value for name, value in assignments.items() if "yard" in name]
        gate_coords = [value for name, value in assignments.items() if "gate" in name]

        if not berth_coords or not yard_coords or not gate_coords:
            raise ValueError("Assignments must include berth, yard, and gate facilities")

        berth_yard = [
            self._euclidean_distance(berth, yard)
            for berth in berth_coords
            for yard in yard_coords
        ]
        yard_gate = [
            self._euclidean_distance(yard, gate)
            for yard in yard_coords
            for gate in gate_coords
        ]

        avg_berth_yard = float(np.mean(berth_yard))
        avg_yard_gate = float(np.mean(yard_gate))
        distance_sum = avg_berth_yard + avg_yard_gate
        throughput_score = 1000.0 / distance_sum if distance_sum > 0 else 0.0
        transport_efficiency = 100.0 / (distance_sum + 1.0)

        return {
            "throughput_score": throughput_score,
            "transport_efficiency": transport_efficiency,
            "avg_berth_yard_distance": avg_berth_yard,
            "avg_yard_gate_distance": avg_yard_gate,
            "land_utilization": len(assignments) / 20.0,
            "estimated_cost": avg_berth_yard * 10.0 + avg_yard_gate * 8.0,
        }

    def _calculate_qualitative_factors(
        self, assignments: Dict[str, Tuple[float, float]]
    ) -> Dict[str, float]:
        """Calculate deterministic geometric proxies, not certified assessments."""
        facilities = list(assignments.values())
        pair_distances = [
            self._euclidean_distance(facilities[i], facilities[j])
            for i in range(len(facilities))
            for j in range(i + 1, len(facilities))
        ]
        minimum_separation = min(pair_distances) if pair_distances else 0.0
        safety_score = min(100.0, minimum_separation)

        inside_buffer = sum(
            self.edge_buffer <= x <= self.terminal_width - self.edge_buffer
            and self.edge_buffer <= y <= self.terminal_length - self.edge_buffer
            for x, y in facilities
        )
        flexibility_score = 100.0 * inside_buffer / len(facilities)

        setback_compliant = sum(
            self.edge_buffer <= x <= self.terminal_width - self.edge_buffer
            and self.edge_buffer <= y <= self.terminal_length - self.edge_buffer
            for x, y in facilities
        )
        regulatory_score = 100.0 * setback_compliant / len(facilities)

        return {
            "safety_score": safety_score,
            "flexibility_score": flexibility_score,
            "environmental_score": 80.0,
            "regulatory_compliance": regulatory_score,
        }

    @staticmethod
    def _calculate_ai_confidence(
        quantitative: Dict[str, float], qualitative: Dict[str, float]
    ) -> float:
        """Combine selected normalized proxies into a descriptive score."""
        normalized_throughput = min(1.0, quantitative["throughput_score"] / 100.0)
        normalized_safety = qualitative["safety_score"] / 100.0
        normalized_efficiency = quantitative["transport_efficiency"] / 100.0
        return (
            0.4 * normalized_throughput
            + 0.3 * normalized_safety
            + 0.3 * normalized_efficiency
        )

    def generate_explanation(self, solution: LayoutSolution) -> Dict[str, Any]:
        """Describe selected indicators and the limits of their interpretation."""
        quantitative = solution.quantitative_metrics
        qualitative = solution.qualitative_factors
        reasoning = [
            (
                "Mean berth-to-yard reference-point distance is "
                f"{quantitative['avg_berth_yard_distance']:.1f} m."
            ),
            (
                "Mean yard-to-gate reference-point distance is "
                f"{quantitative['avg_yard_gate_distance']:.1f} m."
            ),
            (
                "The safety proxy is based on minimum facility-point separation; "
                "it does not represent aisle width or a safety certification."
            ),
        ]
        return {
            "primary_objective": "expert_weighted_candidate_ranking",
            "key_factors": ["distance_proxies", "expert_preferences"],
            "reasoning_steps": reasoning,
            "tradeoffs": [
                "The indicators omit route networks, capacity, costs, and site-specific constraints."
            ],
            "confidence_breakdown": {
                "throughput_contribution": 0.4 * min(1.0, quantitative["throughput_score"] / 100.0),
                "safety_contribution": 0.3 * qualitative["safety_score"] / 100.0,
                "efficiency_contribution": 0.3 * quantitative["transport_efficiency"] / 100.0,
            },
        }


print("Candidate generation and indicator calculations are ready.")

## 3. Ranking candidates using expert preferences

A weighted-sum score provides a transparent way to combine several criteria after expressing them on comparable scales. For candidate $s$ and expert weights $w_k$, this notebook uses

$$
R(s) = \sum_{k \in K} w_k\, z_k(s),
$$

where $K$ is the set of criteria represented in the expert's weight dictionary and $z_k(s)$ is the corresponding normalized indicator. Higher throughput, safety, flexibility, environmental, and operational-efficiency proxies are treated as better; lower estimated cost is converted to a higher cost-efficiency score. The ranking is specific to these criteria, normalizations, and weights. It is not a proof that the highest-ranked candidate is universally best.

The feature-importance display uses the weighted contributions for a selected candidate and normalizes them to sum to one. These values explain the chosen scoring rule; they are not model-derived causal effects.

In [ ]:
class HumanAICollaborationSystem:
    """Coordinate candidate ranking, expert feedback, and illustrative refinement."""

    def __init__(self):
        self.ai_optimizer = ExplainableAIOptimizer()
        self.experts = self._initialize_experts()
        self.collaboration_history: List[CollaborationSession] = []
        self.learning_data: List[Dict[str, Any]] = []

    @staticmethod
    def _initialize_experts() -> List[HumanExpert]:
        """Create example expert profiles whose weights sum to one."""
        return [
            HumanExpert(
                name="Dr. Schmidt",
                expertise_areas=["safety", "operations"],
                experience_years=25,
                decision_weights={
                    "throughput": 0.3,
                    "safety": 0.4,
                    "cost": 0.2,
                    "flexibility": 0.1,
                },
            ),
            HumanExpert(
                name="Ms. Chen",
                expertise_areas=["environmental", "regulatory"],
                experience_years=20,
                decision_weights={
                    "throughput": 0.2,
                    "environmental": 0.3,
                    "regulatory": 0.3,
                    "cost": 0.2,
                },
            ),
            HumanExpert(
                name="Mr. Rodriguez",
                expertise_areas=["operations", "maintenance"],
                experience_years=30,
                decision_weights={
                    "throughput": 0.4,
                    "operational_efficiency": 0.3,
                    "maintenance": 0.2,
                    "flexibility": 0.1,
                },
            ),
        ]

    def start_collaboration_session(
        self, expert: HumanExpert, project_requirements: Dict[str, Any]
    ) -> CollaborationSession:
        """Create a session that keeps its expert and feedback together."""
        session = CollaborationSession(
            session_id=f"session_{len(self.collaboration_history) + 1}",
            human_expert=expert,
            ai_recommendations=[],
            human_feedback=[],
        )
        self.collaboration_history.append(session)
        print(f"Starting {session.session_id} with {expert.name}.")
        print(f"Project requirements: {project_requirements}")
        return session

    def generate_ai_recommendations(
        self,
        session: CollaborationSession,
        requirements: Dict[str, Any],
    ) -> List[AIRecommendation]:
        """Generate candidates, rank them, and explain the leading options."""
        del requirements  # Requirements are displayed in the session and are not modeled as constraints.
        solutions = self.ai_optimizer.generate_ai_solutions(num_solutions=15)
        ranked = self._rank_solutions_for_expert(solutions, session.human_expert)
        recommendations = []

        for solution in ranked[:3]:
            explanation = self.ai_optimizer.generate_explanation(solution)
            solution.explanation = explanation
            recommendations.append(
                AIRecommendation(
                    solution=solution,
                    primary_objective=explanation["primary_objective"],
                    feature_importance=self._calculate_feature_importance(
                        solution, session.human_expert
                    ),
                    reasoning=explanation["reasoning_steps"],
                    alternatives=ranked[3:6],
                    confidence_score=solution.ai_confidence,
                )
            )

        session.ai_recommendations = recommendations
        for index, recommendation in enumerate(recommendations, start=1):
            print(
                f"Recommendation {index}: "
                f"confidence score = {recommendation.confidence_score:.3f}"
            )
        return recommendations

    @staticmethod
    def _rank_solutions_for_expert(
        solutions: List[LayoutSolution], expert: HumanExpert
    ) -> List[LayoutSolution]:
        """Sort candidates by the expert's weighted normalized indicators."""
        def score(solution: LayoutSolution) -> float:
            quantitative = solution.quantitative_metrics
            qualitative = solution.qualitative_factors
            values = {
                "throughput": min(1.0, quantitative["throughput_score"] / 100.0),
                "safety": qualitative["safety_score"] / 100.0,
                "cost": 1.0 - min(1.0, quantitative["estimated_cost"] / 10000.0),
                "flexibility": qualitative["flexibility_score"] / 100.0,
                "environmental": qualitative["environmental_score"] / 100.0,
                "regulatory": qualitative["regulatory_compliance"] / 100.0,
                "operational_efficiency": min(
                    1.0, quantitative["transport_efficiency"] / 100.0
                ),
                "maintenance": qualitative["flexibility_score"] / 100.0,
            }
            return sum(
                weight * values[criterion]
                for criterion, weight in expert.decision_weights.items()
                if criterion in values
            )

        return sorted(solutions, key=score, reverse=True)

    @staticmethod
    def _calculate_feature_importance(
        solution: LayoutSolution, expert: HumanExpert
    ) -> Dict[str, float]:
        """Normalize the expert-weighted criterion contributions for display."""
        quantitative = solution.quantitative_metrics
        qualitative = solution.qualitative_factors
        values = {
            "throughput": min(1.0, quantitative["throughput_score"] / 100.0),
            "safety": qualitative["safety_score"] / 100.0,
            "cost": 1.0 - min(1.0, quantitative["estimated_cost"] / 10000.0),
            "flexibility": qualitative["flexibility_score"] / 100.0,
            "environmental": qualitative["environmental_score"] / 100.0,
            "regulatory": qualitative["regulatory_compliance"] / 100.0,
            "operational_efficiency": min(
                1.0, quantitative["transport_efficiency"] / 100.0
            ),
            "maintenance": qualitative["flexibility_score"] / 100.0,
        }
        contributions = {
            criterion: weight * values[criterion]
            for criterion, weight in expert.decision_weights.items()
            if criterion in values
        }
        total = sum(contributions.values())
        if total == 0.0:
            return {criterion: 0.0 for criterion in contributions}
        return {criterion: value / total for criterion, value in contributions.items()}

    def collect_human_feedback(
        self,
        session: CollaborationSession,
        recommendation: AIRecommendation,
        feedback: Dict[str, Any],
    ) -> None:
        """Record the expert's stated concerns and initial acceptance score."""
        if not 0.0 <= feedback.get("acceptance_level", 0.5) <= 1.0:
            raise ValueError("acceptance_level must be between 0 and 1")
        if recommendation not in session.ai_recommendations:
            raise ValueError("The recommendation must belong to this session")

        acceptance = float(feedback.get("acceptance_level", 0.5))
        recommendation.solution.human_acceptance = acceptance
        record = {
            "timestamp": time.time(),
            "feedback": feedback.copy(),
            "expert_concerns": list(feedback.get("concerns", [])),
            "suggested_modifications": list(feedback.get("modifications", [])),
            "acceptance_level": acceptance,
        }
        session.human_feedback.append(record)
        self.learning_data.append(record.copy())
        print(f"Recorded initial acceptance: {acceptance:.2f}")

    def collaborative_refinement(
        self, session: CollaborationSession
    ) -> LayoutSolution:
        """Apply the recorded coordinate rules and recalculate all indicators."""
        if not session.ai_recommendations:
            raise ValueError("Generate recommendations before refinement")

        base_solution = session.ai_recommendations[0].solution
        refined = self._apply_human_modifications(base_solution, session)
        refined.quantitative_metrics = self.ai_optimizer._calculate_quantitative_metrics(
            refined.facility_assignments
        )
        refined.qualitative_factors = self.ai_optimizer._calculate_qualitative_factors(
            refined.facility_assignments
        )
        refined.ai_confidence = self.ai_optimizer._calculate_ai_confidence(
            refined.quantitative_metrics, refined.qualitative_factors
        )
        refined.explanation = self.ai_optimizer.generate_explanation(refined)
        session.final_solution = refined
        return refined

    def _apply_human_modifications(
        self, solution: LayoutSolution, session: CollaborationSession
    ) -> LayoutSolution:
        """Apply only the explicitly represented feedback categories."""
        assignments = solution.facility_assignments.copy()
        for record in session.human_feedback:
            concerns = record["expert_concerns"]
            modifications = record["suggested_modifications"]
            if "safety" in concerns or "narrow_aisles" in concerns:
                assignments = self._increase_facility_separation(assignments)
            if "regulatory" in concerns or "setback_violation" in concerns:
                assignments = self._ensure_regulatory_setbacks(assignments)
            if "operational" in concerns or "access_efficiency" in modifications:
                assignments = self._optimize_access_routes(assignments)

        return LayoutSolution(
            facility_assignments=assignments,
            quantitative_metrics=solution.quantitative_metrics.copy(),
            qualitative_factors=solution.qualitative_factors.copy(),
            ai_confidence=solution.ai_confidence,
            human_acceptance=solution.human_acceptance,
        )

    def _increase_facility_separation(
        self, assignments: Dict[str, Tuple[float, float]]
    ) -> Dict[str, Tuple[float, float]]:
        """Move yard-block reference points 50 m farther from the berth row."""
        refined = assignments.copy()
        for name, (x, y) in refined.items():
            if "yard_block" in name:
                refined[name] = (x, min(self.ai_optimizer.terminal_length, y + 50.0))
        return refined

    def _ensure_regulatory_setbacks(
        self, assignments: Dict[str, Tuple[float, float]]
    ) -> Dict[str, Tuple[float, float]]:
        """Place reference points within the illustrative 50 m edge buffer."""
        refined = {}
        width = self.ai_optimizer.terminal_width
        length = self.ai_optimizer.terminal_length
        setback = self.ai_optimizer.edge_buffer
        for name, (x, y) in assignments.items():
            refined[name] = (
                max(setback, min(width - setback, x)),
                max(setback, min(length - setback, y)),
            )
        return refined

    def _optimize_access_routes(
        self, assignments: Dict[str, Tuple[float, float]]
    ) -> Dict[str, Tuple[float, float]]:
        """Move gate reference points toward the landside edge in the template."""
        refined = assignments.copy()
        landside_y = self.ai_optimizer.terminal_length - self.ai_optimizer.edge_buffer
        for name, (x, y) in refined.items():
            if "gate" in name:
                refined[name] = (x, max(y, landside_y))
        return refined

    @staticmethod
    def evaluate_collaboration_outcome(
        session: CollaborationSession,
    ) -> Dict[str, Any]:
        """Compare proxy indicators and report the recorded acceptance level."""
        if session.final_solution is None or not session.ai_recommendations:
            raise ValueError("A refined solution and recommendation are required")

        original = session.ai_recommendations[0].solution
        refined = session.final_solution
        return {
            "throughput_change": (
                refined.quantitative_metrics["throughput_score"]
                - original.quantitative_metrics["throughput_score"]
            ),
            "safety_proxy_change": (
                refined.qualitative_factors["safety_score"]
                - original.qualitative_factors["safety_score"]
            ),
            "recorded_initial_acceptance": original.human_acceptance,
            "feedback_records": len(session.human_feedback),
        }


print("Expert ranking and feedback workflow are ready.")

## 4. Generate and inspect recommendations

This step applies the candidate-generation and weighted-ranking procedure to an example set of project requirements. The requirements are printed for context, but the code does not convert them into hard constraints. The displayed indicators therefore describe this notebook's simplified model, not a validated terminal design.

In [ ]:
print("HUMAN–AI COLLABORATIVE TERMINAL LAYOUT DESIGN")

collaboration_system = HumanAICollaborationSystem()
safety_expert = collaboration_system.experts[0]

project_requirements = {
    "objective": "maximize throughput subject to expert review",
    "constraints_for_review": [
        "safety",
        "regulatory compliance",
        "environmental protection",
    ],
    "budget_limit": 10_000_000,
    "timeline": "24 months",
    "context": "Illustrative container-terminal setting",
}

session = collaboration_system.start_collaboration_session(
    safety_expert, project_requirements
)
recommendations = collaboration_system.generate_ai_recommendations(
    session, project_requirements
)
top_recommendation = recommendations[0]

print("\nTop-ranked candidate indicators:")
print(
    "  Throughput proxy: "
    f"{top_recommendation.solution.quantitative_metrics['throughput_score']:.4f}"
)
print(
    "  Transport-efficiency proxy: "
    f"{top_recommendation.solution.quantitative_metrics['transport_efficiency']:.4f}"
)
print(
    "  Safety proxy: "
    f"{top_recommendation.solution.qualitative_factors['safety_score']:.1f} / 100"
)
print(
    "  Estimated-cost proxy: "
    f"{top_recommendation.solution.quantitative_metrics['estimated_cost']:.1f}"
)
print("\nExplanation:")
for reason in top_recommendation.reasoning:
    print(f"  - {reason}")

assert len(recommendations) == 3
assert all(0.0 <= item.confidence_score <= 1.0 for item in recommendations)

## 5. Record human feedback

An expert can identify concerns that are absent from the simplified metrics. Here, the feedback is an illustrative input describing narrow-aisle and access concerns. Because aisle widths and wind conditions are not represented in the coordinate model, these concerns are not inferred from the candidate's calculated scores. The recorded acceptance is an initial review value; the notebook does not claim that it is a post-refinement approval.

In [ ]:
human_feedback = {
    "acceptance_level": 0.6,
    "concerns": [
        "narrow_aisles",
        "safety_risk_high_wind",
        "maintenance_access_difficulty",
    ],
    "modifications": [
        "increase_aisle_width",
        "improve_wind_protection",
        "enhance_maintenance_access",
    ],
    "expert_reasoning": [
        "Aisle width and wind exposure require site-specific engineering review.",
        "Maintenance access should be assessed against vehicle and emergency requirements.",
    ],
}

collaboration_system.collect_human_feedback(
    session, top_recommendation, human_feedback
)

print(f"Expert: {safety_expert.name}")
print(f"Initial acceptance: {human_feedback['acceptance_level']:.2f}")
print(f"Concerns: {', '.join(human_feedback['concerns'])}")
print("Expert reasoning:")
for reason in human_feedback["expert_reasoning"]:
    print(f"  - {reason}")

## 6. Apply feedback-informed refinements

The refinement function starts from the highest-ranked candidate and applies the coordinate rules associated with recorded concerns. In this example, a narrow-aisle concern shifts yard-block reference points away from the berth row. This adjustment is a transparent demonstration of feedback integration, not a verified aisle-design rule. The metrics are recalculated from the refined coordinates so that reported values correspond to the displayed layout.

In [ ]:
refined_solution = collaboration_system.collaborative_refinement(session)
original_solution = top_recommendation.solution

print("Refined candidate indicators:")
print(
    "  Throughput proxy: "
    f"{refined_solution.quantitative_metrics['throughput_score']:.4f}"
)
print(
    "  Transport-efficiency proxy: "
    f"{refined_solution.quantitative_metrics['transport_efficiency']:.4f}"
)
print(
    "  Safety proxy: "
    f"{refined_solution.qualitative_factors['safety_score']:.1f} / 100"
)
print(
    "  Flexibility proxy: "
    f"{refined_solution.qualitative_factors['flexibility_score']:.1f} / 100"
)
print(
    "  Estimated-cost proxy: "
    f"{refined_solution.quantitative_metrics['estimated_cost']:.1f}"
)

for name, coordinate in refined_solution.facility_assignments.items():
    x, y = coordinate
    assert 0.0 <= x <= 1000.0
    assert 0.0 <= y <= 800.0

print("\nCoordinate-bound checks passed for all facilities.")

## 7. Evaluate the modeled collaboration outcome

The comparison reports changes in the notebook's throughput and safety proxies, not operational improvements. A change can be positive or negative because feedback-driven coordinate adjustments may trade one modeled indicator against another. Acceptance is reported as the initial expert-provided value; a separate review would be needed to assess acceptance after refinement.

In [ ]:
outcome = collaboration_system.evaluate_collaboration_outcome(session)

print("MODELED COLLABORATION OUTCOME")
print(f"Throughput-proxy change: {outcome['throughput_change']:+.4f}")
print(f"Safety-proxy change: {outcome['safety_proxy_change']:+.1f} points")
print(
    "Recorded initial acceptance: "
    f"{outcome['recorded_initial_acceptance']:.2f}"
)
print(f"Feedback records: {outcome['feedback_records']}")

assert np.isclose(
    refined_solution.quantitative_metrics["throughput_score"]
    - original_solution.quantitative_metrics["throughput_score"],
    outcome["throughput_change"],
)
assert np.isclose(
    refined_solution.qualitative_factors["safety_score"]
    - original_solution.qualitative_factors["safety_score"],
    outcome["safety_proxy_change"],
)

## 8. Visualize candidates, criteria, and feedback

The plots show the quantities calculated by the notebook: criterion proxies, weighted contributions, and facility reference-point positions. The facility-type legend explains the color coding. Coordinates are plotted in meters, and labels identify each facility. The plots do not encode aisle widths, wind exposure, routes, or stakeholder survey results because those data are not part of the model.

In [ ]:
def visualize_human_ai_collaboration(
    original: LayoutSolution,
    refined: LayoutSolution,
    recommendation: AIRecommendation,
    feedback: Dict[str, Any],
) -> None:
    """Plot modeled indicators, weighted contributions, and layout coordinates."""
    figure, axes = plt.subplots(2, 2, figsize=(15, 11))
    ax_metrics, ax_importance, ax_original, ax_refined = axes.ravel()

    categories = [
        "Throughput proxy",
        "Safety proxy",
        "Flexibility proxy",
        "Regulatory proxy",
        "Environmental proxy",
    ]
    original_values = [
        original.quantitative_metrics["throughput_score"],
        original.qualitative_factors["safety_score"],
        original.qualitative_factors["flexibility_score"],
        original.qualitative_factors["regulatory_compliance"],
        original.qualitative_factors["environmental_score"],
    ]
    refined_values = [
        refined.quantitative_metrics["throughput_score"],
        refined.qualitative_factors["safety_score"],
        refined.qualitative_factors["flexibility_score"],
        refined.qualitative_factors["regulatory_compliance"],
        refined.qualitative_factors["environmental_score"],
]
    positions = np.arange(len(categories))
    width = 0.36
    ax_metrics.bar(
        positions - width / 2,
        original_values,
        width,
        label="Ranked candidate",
        color="steelblue",
    )
    ax_metrics.bar(
        positions + width / 2,
        refined_values,
        width,
        label="Feedback-refined candidate",
        color="darkorange",
    )
    ax_metrics.set_xticks(positions)
    ax_metrics.set_xticklabels(categories, rotation=35, ha="right")
    ax_metrics.set_ylabel("Indicator value (proxy scale)")
    ax_metrics.set_title("Candidate indicator comparison")
    ax_metrics.legend()
    ax_metrics.grid(axis="y", alpha=0.25)

    importance = recommendation.feature_importance
    ax_importance.barh(list(importance), list(importance.values()), color="slateblue")
    ax_importance.set_xlabel("Normalized weighted contribution")
    ax_importance.set_title("Expert-weighted contributions")
    ax_importance.grid(axis="x", alpha=0.25)

    facility_styles = {
        "berth": ("royalblue", "s"),
        "yard": ("seagreen", "o"),
        "gate": ("firebrick", "^"),
        "support": ("darkorange", "D"),
    }

    def draw_layout(ax: Any, solution: LayoutSolution, title: str) -> None:
        for name, (x, y) in solution.facility_assignments.items():
            if "berth" in name:
                category = "berth"
            elif "yard" in name:
                category = "yard"
            elif "gate" in name:
                category = "gate"
            else:
                category = "support"
            color, marker = facility_styles[category]
            ax.scatter(
                x,
                y,
                color=color,
                marker=marker,
                s=65,
                label=category.title(),
            )
            ax.annotate(name, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=7)

        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), title="Facility type", fontsize=8)
        ax.set_xlim(0, 1000)
        ax.set_ylim(0, 800)
        ax.set_xlabel("X coordinate (m)")
        ax.set_ylabel("Y coordinate (m)")
        ax.set_title(title)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.25)

    draw_layout(ax_original, original, "Ranked candidate layout")
    draw_layout(ax_refined, refined, "Feedback-refined layout")
    figure.suptitle(
        "Human–AI layout review — initial acceptance "
        f"{feedback.get('acceptance_level', 0.5):.2f}",
        fontsize=14,
    )
    figure.tight_layout()
    plt.show()


visualize_human_ai_collaboration(
    original_solution,
    refined_solution,
    top_recommendation,
    human_feedback,
)

## 9. Demonstrate outcome recording across sessions

Repeated sessions can be used to collect examples of expert choices and feedback. The demonstration below records a small set of illustrative sessions with the same explicit feedback rule. It calculates descriptive summaries from the resulting records, but it does not fit a learning algorithm, change expert weights, or establish that future recommendations improve. Demonstrating data collection is distinct from demonstrating model learning.

In [ ]:
def demonstrate_learning_system(
    system: HumanAICollaborationSystem,
    requirements: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """Record illustrative sessions without training or updating a model."""
    records = []
    feedback_template = {
        "acceptance_level": 0.6,
        "concerns": ["safety"],
        "modifications": [],
    }

    for session_number in range(1, 6):
        expert = system.experts[(session_number - 1) % len(system.experts)]
        session_requirements = requirements.copy()
        session_requirements["context"] = f"Illustrative variant {session_number}"
        learning_session = system.start_collaboration_session(
            expert, session_requirements
        )
        session_recommendations = system.generate_ai_recommendations(
            learning_session, session_requirements
        )
        system.collect_human_feedback(
            learning_session,
            session_recommendations[0],
            feedback_template.copy(),
        )
        refined = system.collaborative_refinement(learning_session)
        outcome_record = system.evaluate_collaboration_outcome(learning_session)
        records.append(
            {
                "session": session_number,
                "expert": expert.name,
                "recorded_acceptance": outcome_record[
                    "recorded_initial_acceptance"
                ],
                "safety_proxy": refined.qualitative_factors["safety_score"],
                "throughput_proxy": refined.quantitative_metrics[
                    "throughput_score"
                ],
            }
        )

    acceptance_values = [record["recorded_acceptance"] for record in records]
    safety_values = [record["safety_proxy"] for record in records]
    throughput_values = [record["throughput_proxy"] for record in records]

    print("Session records:")
    for record in records:
        print(
            f"  Session {record['session']} ({record['expert']}): "
            f"acceptance={record['recorded_acceptance']:.2f}, "
            f"safety proxy={record['safety_proxy']:.1f}, "
            f"throughput proxy={record['throughput_proxy']:.4f}"
        )

    print("\nDescriptive summaries of these illustrative records:")
    print(f"  Mean recorded acceptance: {np.mean(acceptance_values):.3f}")
    print(f"  Mean safety proxy: {np.mean(safety_values):.2f}")
    print(f"  Mean throughput proxy: {np.mean(throughput_values):.4f}")
    print("  No predictive model is trained or updated by this demonstration.")
    return records


learning_results = demonstrate_learning_system(
    collaboration_system, project_requirements
)

## 10. Interpretation, strengths, and limitations

### Why include a human review stage?

A ranking system can only use information represented in its inputs and scoring rules. Domain experts can raise concerns about operational context, safety, regulations, or stakeholder needs that the simplified coordinate indicators do not encode. The review stage makes those concerns explicit and connects them to transparent, limited coordinate adjustments.

### Strengths of this workflow

- Candidate indicators, expert weights, and coordinate changes are visible and inspectable.
- The ranking makes the influence of stated preferences explicit.
- Feedback is stored with the session, and the refined layout's indicators are recalculated from its coordinates.
- The workflow distinguishes calculated proxies from engineering validation and empirical evidence.

### Limitations

- Candidate generation explores only a small, hand-designed family of layouts; the ranking is not a global optimization result.
- Straight-line distances ignore roads, routes, traffic, capacity, congestion, handling equipment, and facility footprints.
- The safety, flexibility, environmental, regulatory, cost, and throughput indicators are simplified proxies and are not calibrated or certified.
- The feedback rules are illustrative and do not model all possible expert judgments or guarantee that a requested concern has been resolved.
- The session-recording demonstration does not learn from past feedback or measure improvement over time.
- No stakeholder-satisfaction survey, decision-time study, or real terminal case data is included.

## Summary

This notebook presents an illustrative human–AI collaboration workflow for terminal-layout decision support. It represents candidate layouts and expert profiles, calculates explicit geometric proxies, ranks candidates with an expert-weighted sum, records human concerns, applies specified coordinate refinements, and compares recalculated indicators.

The main lesson is methodological: a recommendation is only as reliable as its candidate set, data, metrics, assumptions, and validation. Human review can expose relevant information that a simplified model omits, but recording feedback or changing coordinates does not by itself establish safety, compliance, improved throughput, stakeholder acceptance, or model learning. Those claims require appropriate engineering analysis and empirical evidence.